# SK-8-MCP : Model Context Protocol et Semantic Kernel en C#

**Navigation** : [Python](08-SemanticKernel-MCP.ipynb) | [Index](README.md) | [07 MultiModal](07-SemanticKernel-MultiModal.ipynb)

## Objectifs d’apprentissage

- Découvrir les outils et leurs schémas via un vrai serveur MCP stdio.
- Appeler et valider prix, TVA, chemin et texte, puis distinguer erreur de l’outil et panne de transport.
- Connecter les outils MCP à Semantic Kernel, router sans modèle et exposer une fonction SK en MCP.

**Prérequis** : C#, .NET SDK 9+, notebooks SK précédents et accès NuGet lors de la première restauration. **Durée estimée** : 50 minutes. Aucun modèle ni clé API ne sont nécessaires. Exécuter depuis le dossier de ce notebook.

## 1. Architecture

MCP normalise les primitives *tools*, *resources* et *prompts*. Ici un client lance un serveur en sous-processus et échange des messages JSON-RPC sur stdio : `initialize`, `tools/list`, puis `tools/call`. Les fonctions métier sont locales au serveur ; le protocole les rend interopérables. Le SDK officiel C# et Semantic Kernel sont épinglés ci-dessous.

In [1]:
#r "nuget: ModelContextProtocol, 2.4.0"
#r "nuget: Microsoft.SemanticKernel, 1.82.0"
using System;
using System.IO;
using System.Linq;
using System.Collections.Generic;
using System.Diagnostics;
using System.Text.Json;
using System.Threading;
using System.Threading.Tasks;
using Microsoft.SemanticKernel;
using ModelContextProtocol.Client;
Console.WriteLine("SDK chargés : ModelContextProtocol 2.4.0 ; Microsoft.SemanticKernel 1.82.0");

Installed Packages Microsoft.SemanticKernel, 1.82.0 ModelContextProtocol, 2.4.0

SDK chargés : ModelContextProtocol 2.4.0 ; Microsoft.SemanticKernel 1.82.0


## 2. Construire le serveur

Le projet voisin expose cinq outils : prix TTC, détail TVA, TVA inverse, vérification de chemin et analyse de chaîne. Il réserve stdout à JSON-RPC. La construction précède le lancement ; lire [son code](sk08-mcp-server/Program.cs) et sa [documentation](sk08-mcp-server/README.md).

In [2]:
var project = Path.GetFullPath("sk08-mcp-server/Sk08McpServer.csproj");
var dll = Path.GetFullPath("sk08-mcp-server/bin/Debug/net9.0/Sk08McpServer.dll");
var build = Process.Start(new ProcessStartInfo("dotnet") { ArgumentList = { "build", project, "--nologo" }, RedirectStandardOutput = true, RedirectStandardError = true, UseShellExecute = false });
var buildOutTask = build!.StandardOutput.ReadToEndAsync();
var buildErrTask = build.StandardError.ReadToEndAsync();
try { await build.WaitForExitAsync().WaitAsync(TimeSpan.FromMinutes(2)); }
catch (TimeoutException) { build.Kill(entireProcessTree: true); await build.WaitForExitAsync(); throw; }
var buildOut = await buildOutTask;
var buildErr = await buildErrTask;
if (build.ExitCode != 0) throw new Exception(buildOut + buildErr);
Console.WriteLine($"Serveur compilé : {Path.GetFileName(dll)}, code {build.ExitCode}");
async Task AvecClient(Func<McpClient, CancellationToken, Task> action, bool exporterSk = false)
{
    using var limite = new CancellationTokenSource(TimeSpan.FromSeconds(30));
    var options = new StdioClientTransportOptions { Command = "dotnet", Arguments = exporterSk ? new[] { dll, "--sk" } : new[] { dll }, ShutdownTimeout = TimeSpan.FromSeconds(5) };
    await using var session = await McpClient.CreateAsync(new StdioClientTransport(options), cancellationToken: limite.Token);
    await action(session, limite.Token);
}

Serveur compilé : Sk08McpServer.dll, code 0


Le binaire est lancé par le transport du SDK. `CreateAsync` effectue l’initialisation ; la liste des outils est demandée au serveur, avec les schémas d’entrée dérivés des signatures.

In [3]:
await AvecClient(async (client, ct) =>
{
    var tools = await client.ListToolsAsync(cancellationToken: ct);
    Console.WriteLine($"initialize : {client.ServerInfo?.Name}, protocole {client.NegotiatedProtocolVersion}");
    Console.WriteLine($"tools/list : {tools.Count} outils");
    foreach (var tool in tools) Console.WriteLine($"{tool.Name} : {tool.Description}\n  entrée : {tool.JsonSchema}\n  sortie : {tool.ProtocolTool.OutputSchema}");
});

initialize : Sk08McpServer, protocole 2026-07-28


tools/list : 5 outils


prix_ttc : Calcule le prix TTC après remise et TVA de 20%.
  entrée : {"type":"object","properties":{"base":{"description":"Prix HT strictement positif","type":"number"},"remise_pct":{"description":"Remise de 0 \u00E0 100 pour cent","type":"integer","default":0}},"required":["base"]}
  sortie : {"type":"object","properties":{"ht":{"type":"number"},"ttc":{"type":"number"}},"required":["ht","ttc"]}


calcule_tva : Détaille la TVA à 20%.
  entrée : {"type":"object","properties":{"montant_ht":{"type":"number"}},"required":["montant_ht"]}
  sortie : {"type":"object","properties":{"ht":{"type":"number"},"tva":{"type":"number"},"ttc":{"type":"number"}},"required":["ht","tva","ttc"]}


tva_inverse : Retrouve le HT depuis le TTC et le taux.
  entrée : {"type":"object","properties":{"montant_ttc":{"type":"number"},"taux_tva":{"type":"number"}},"required":["montant_ttc","taux_tva"]}
  sortie : {"type":"object","properties":{"ttc":{"type":"number"},"taux_tva":{"type":"number"},"ht":{"type":"number"}},"required":["ttc","taux_tva","ht"]}


verifie_chemin : Teste si un chemin se situe dans une zone autorisée (sans résoudre les liens symboliques).
  entrée : {"type":"object","properties":{"chemin":{"type":"string"},"zone_autorisee":{"type":"string"}},"required":["chemin","zone_autorisee"]}
  sortie : {"type":"object","properties":{"chemin":{"type":"string"},"zone_autorisee":{"type":"string"},"dans_zone":{"type":"boolean"}},"required":["chemin","zone_autorisee","dans_zone"]}


analyse_chaine : Compte caractères et mots et donne le premier mot.
  entrée : {"type":"object","properties":{"texte":{"type":"string"}},"required":["texte"]}
  sortie : {"type":"object","properties":{"nb_caracteres":{"type":"integer"},"nb_mots":{"type":"integer"},"premier_mot":{"type":"string"}},"required":["nb_caracteres","nb_mots","premier_mot"]}


La propriété `required` décrit les paramètres obligatoires ; le défaut de remise est visible. Les contraintes numériques sont contrôlées dans le corps du serveur C# et non exprimées comme les bornes Pydantic du notebook Python. Les records de retour et `UseStructuredContent=true` produisent un schéma de sortie et un `structuredContent` en plus du texte lisible.

In [4]:
await AvecClient(async (client, ct) =>
{
    var prix = await client.CallToolAsync("prix_ttc", new Dictionary<string, object> { ["base"] = 100.0, ["remise_pct"] = 10 }, cancellationToken: ct);
    var tva = await client.CallToolAsync("calcule_tva", new Dictionary<string, object> { ["montant_ht"] = 1000.0 }, cancellationToken: ct);
    var inverse = await client.CallToolAsync("tva_inverse", new Dictionary<string, object> { ["montant_ttc"] = 850.0, ["taux_tva"] = 0.196 }, cancellationToken: ct);
    Console.WriteLine("prix : " + JsonSerializer.Serialize(prix));
    Console.WriteLine("TVA : " + JsonSerializer.Serialize(tva));
    Console.WriteLine("TVA inverse : " + JsonSerializer.Serialize(inverse));
    var analyse = await client.CallToolAsync("analyse_chaine", new Dictionary<string, object> { ["texte"] = "Le Model Context Protocol normalise les schemas d'outils." }, cancellationToken: ct);
    Console.WriteLine("Analyse structurée : " + JsonSerializer.Serialize(analyse.StructuredContent));
    var ttcCalcule = JsonSerializer.SerializeToElement(prix.StructuredContent).GetProperty("ttc").GetDouble();
    var retourInverse = await client.CallToolAsync("tva_inverse", new Dictionary<string, object> { ["montant_ttc"] = ttcCalcule, ["taux_tva"] = 0.2 }, cancellationToken: ct);
    Console.WriteLine($"Enchaînement TTC {ttcCalcule} -> HT : {JsonSerializer.Serialize(retourInverse.StructuredContent)}");
    var petit = await client.CallToolAsync("prix_ttc", new Dictionary<string, object> { ["base"] = 0.03, ["remise_pct"] = 10 }, cancellationToken: ct);
    var p = JsonSerializer.SerializeToElement(petit.StructuredContent);
    if (p.GetProperty("ht").GetDouble() != 0.03 || p.GetProperty("ttc").GetDouble() != 0.03) throw new Exception("Arrondi HT/TTC non conforme");
    Console.WriteLine("Cas d'arrondi 0,03 remise 10 % : " + JsonSerializer.Serialize(petit.StructuredContent));
    var emoji = await client.CallToolAsync("analyse_chaine", new Dictionary<string, object> { ["texte"] = "😀" }, cancellationToken: ct);
    var e = JsonSerializer.SerializeToElement(emoji.StructuredContent);
    if (e.GetProperty("nb_caracteres").GetInt32() != 1) throw new Exception("Comptage Unicode non conforme");
    Console.WriteLine("Caractères Unicode : " + JsonSerializer.Serialize(emoji.StructuredContent));
});

prix : {"content":[{"type":"text","text":"{\u0022ht\u0022:90,\u0022ttc\u0022:108}"}],"structuredContent":{"ht":90,"ttc":108},"isError":null,"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


TVA : {"content":[{"type":"text","text":"{\u0022ht\u0022:1000,\u0022tva\u0022:200,\u0022ttc\u0022:1200}"}],"structuredContent":{"ht":1000,"tva":200,"ttc":1200},"isError":null,"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


TVA inverse : {"content":[{"type":"text","text":"{\u0022ttc\u0022:850,\u0022taux_tva\u0022:0.196,\u0022ht\u0022:710.7023}"}],"structuredContent":{"ttc":850,"taux_tva":0.196,"ht":710.7023},"isError":null,"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


Analyse structurée : {"nb_caracteres":57,"nb_mots":8,"premier_mot":"Le"}


Enchaînement TTC 108 -> HT : {"ttc":108,"taux_tva":0.2,"ht":90}


Cas d'arrondi 0,03 remise 10 % : {"ht":0.03,"ttc":0.03}


Caractères Unicode : {"nb_caracteres":1,"nb_mots":1,"premier_mot":"\uD83D\uDE00"}


Les résultats proviennent de `tools/call` ; le prix est calculé après remise puis TVA. Le TTC issu du premier appel est relu dans `structuredContent` et transmis à la TVA inverse. Pour les montants réels, préférer `decimal` et une politique d’arrondi métier explicitée.

### Exercice 1 — Router une demande

Choisir un nom d’outil et ses arguments pour une demande de prix de 1000 HT avec 25 % de remise. Puis ouvrir une session avec `AvecClient` et invoquer `client.CallToolAsync`. Le squelette compile avant votre intervention.

In [5]:
string demande = "Prix TTC pour 1000 HT avec 25 % de remise";
string outilChoisi = null; // TODO : choisir parmi les outils découverts
var argumentsEx1 = new Dictionary<string, object>(); // TODO : renseigner base et remise_pct
Console.WriteLine($"Exercice à compléter : {demande}; outil={outilChoisi ?? "non choisi"}, arguments={argumentsEx1.Count}");

Exercice à compléter : Prix TTC pour 1000 HT avec 25 % de remise; outil=non choisi, arguments=0


## 3. Validation et erreurs

Un argument absent ou une valeur hors bornes atteint le serveur et produit une erreur d’appel. Un nom inconnu peut produire une erreur JSON-RPC plutôt qu’un résultat `isError`. Une panne de lancement est une exception de transport distincte.

In [6]:
await AvecClient(async (client, ct) =>
{
var absent = await client.CallToolAsync("prix_ttc", new Dictionary<string, object>(), cancellationToken: ct);
var horsBornes = await client.CallToolAsync("prix_ttc", new Dictionary<string, object> { ["base"] = 100.0, ["remise_pct"] = 150 }, cancellationToken: ct);
var negatif = await client.CallToolAsync("calcule_tva", new Dictionary<string, object> { ["montant_ht"] = -10.0 }, cancellationToken: ct);
Console.WriteLine($"base absente : isError={absent.IsError}, {JsonSerializer.Serialize(absent.Content)}");
Console.WriteLine($"remise 150 : isError={horsBornes.IsError}, {JsonSerializer.Serialize(horsBornes.Content)}");
Console.WriteLine($"HT négatif : isError={negatif.IsError}, {JsonSerializer.Serialize(negatif.Content)}");
try { var unknown = await client.CallToolAsync("outil_inconnu", new Dictionary<string, object>(), cancellationToken: ct); Console.WriteLine("outil inconnu : " + JsonSerializer.Serialize(unknown)); }
catch (Exception ex) { Console.WriteLine($"outil inconnu : {ex.GetType().Name} : {ex.Message}"); }
});
try { using var limiteTransport = new CancellationTokenSource(TimeSpan.FromSeconds(5)); await using var impossible = await McpClient.CreateAsync(new StdioClientTransport(new() { Command = "sk08-commande-inexistante" }), cancellationToken: limiteTransport.Token); }
catch (Exception ex) { Console.WriteLine($"transport indisponible : {ex.GetType().Name}"); }

base absente : isError=True, [{"type":"text","text":"An error occurred invoking \u0027prix_ttc\u0027."}]


remise 150 : isError=True, [{"type":"text","text":"An error occurred invoking \u0027prix_ttc\u0027."}]


HT négatif : isError=True, [{"type":"text","text":"An error occurred invoking \u0027calcule_tva\u0027."}]


outil inconnu : McpProtocolException : Request failed (remote): Unknown tool: 'outil_inconnu'


transport indisponible : IOException


La validation métier échoue côté serveur ; le message générique ne doit pas être interprété comme une panne du pipe. Le contrôle de chemin ci-dessous est **lexical** : il ne résout pas les liens symboliques et ne constitue pas une frontière de sécurité pour accéder au système de fichiers.

In [7]:
await AvecClient(async (client, ct) =>
{
foreach (var (chemin, zone) in new[] { ("/tmp/data/file.csv", "/tmp/data"), ("/etc/passwd", "/tmp/data"), ("/tmp/data2/file", "/tmp/data"), ("", "/tmp") })
{
    var r = await client.CallToolAsync("verifie_chemin", new Dictionary<string, object> { ["chemin"] = chemin, ["zone_autorisee"] = zone }, cancellationToken: ct);
    Console.WriteLine($"{chemin} dans {zone} : {JsonSerializer.Serialize(r.Content)}");
}
});

/tmp/data/file.csv dans /tmp/data : [{"type":"text","text":"{\u0022chemin\u0022:\u0022/tmp/data/file.csv\u0022,\u0022zone_autorisee\u0022:\u0022/tmp/data\u0022,\u0022dans_zone\u0022:true}"}]


/etc/passwd dans /tmp/data : [{"type":"text","text":"{\u0022chemin\u0022:\u0022/etc/passwd\u0022,\u0022zone_autorisee\u0022:\u0022/tmp/data\u0022,\u0022dans_zone\u0022:false}"}]


/tmp/data2/file dans /tmp/data : [{"type":"text","text":"{\u0022chemin\u0022:\u0022/tmp/data2/file\u0022,\u0022zone_autorisee\u0022:\u0022/tmp/data\u0022,\u0022dans_zone\u0022:false}"}]


 dans /tmp : [{"type":"text","text":"{\u0022chemin\u0022:\u0022\u0022,\u0022zone_autorisee\u0022:\u0022/tmp\u0022,\u0022dans_zone\u0022:false}"}]


Le test de préfixe de répertoire évite de confondre `data` et `data2`. Pour une autorisation effective, il faut aussi traiter les liens symboliques et les courses entre vérification et ouverture.

### Exercice 2 — Contrôle de chemin

Appeler `verifie_chemin` pour un chemin de votre choix et lire le booléen dans le JSON de `structuredContent`. Indice : ouvrir une session avec `AvecClient` ; `client.CallToolAsync` utilise les noms `chemin` et `zone_autorisee`.

In [8]:
string cheminEx2 = "/tmp/data/document.csv";
string zoneEx2 = "/tmp/data";
// TODO : appeler verifie_chemin et lire dans_zone dans son contenu.
Console.WriteLine($"Exercice à compléter : vérifier {cheminEx2} dans {zoneEx2}");

Exercice à compléter : vérifier /tmp/data/document.csv dans /tmp/data


## 4. Un client Semantic Kernel pour MCP

Le SDK .NET expose chaque outil découvert comme `AIFunction`. `AsKernelFunction()` l’adapte à SK ; aucun connecteur `Microsoft.SemanticKernel.Connectors.Mcp` n’est requis. Chaque session reste ouverte pendant les invocations puis `await using` la ferme même en cas d’exception. Une limite de temps borne les appels directs.

In [9]:
await AvecClient(async (client, ct) =>
{
    var tools = await client.ListToolsAsync(cancellationToken: ct);
    var kernel = Kernel.CreateBuilder().Build();
    kernel.Plugins.AddFromFunctions("mcp", tools.Select(t => t.AsKernelFunction()));
    var skResult = await kernel.InvokeAsync("mcp", "prix_ttc", new KernelArguments { ["base"] = 200.0, ["remise_pct"] = 15 }, cancellationToken: ct);
    Console.WriteLine($"SK : {kernel.Plugins["mcp"].Count()} fonctions ; prix_ttc(200,15) = {skResult}");
    var skErreur = await kernel.InvokeAsync("mcp", "prix_ttc", new KernelArguments { ["base"] = 200.0, ["remise_pct"] = 150 }, cancellationToken: ct);
    using var erreurJson = JsonDocument.Parse(skErreur.ToString());
    if (!erreurJson.RootElement.GetProperty("isError").GetBoolean()) throw new Exception("Erreur MCP non propagée dans SK");
    Console.WriteLine("SK : remise 150, résultat MCP isError=" + erreurJson.RootElement.GetProperty("isError"));
});

SK : 5 fonctions ; prix_ttc(200,15) = {"content":[{"type":"text","text":"{\u0022ht\u0022:170,\u0022ttc\u0022:204}"}],"structuredContent":{"ht":170,"ttc":204},"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


SK : remise 150, résultat MCP isError=True



warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.11.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.11.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Ab

La fonction SK délègue à l’outil MCP découvert. Le routage suivant est déterministe : les noms et arguments sont choisis explicitement. Il illustre la sélection d’un outil sans prétendre exécuter `FunctionChoiceBehavior.Auto()` ni une réponse de LLM.

In [10]:
await AvecClient(async (client, ct) =>
{
    var tools = await client.ListToolsAsync(cancellationToken: ct);
    var kernel = Kernel.CreateBuilder().Build();
    kernel.Plugins.AddFromFunctions("mcp", tools.Select(t => t.AsKernelFunction()));
var requetes = new[] {
    ("Détail TVA pour 1000 HT", "calcule_tva", new KernelArguments { ["montant_ht"] = 1000.0 }),
    ("Prix de 500 HT, remise 15 %", "prix_ttc", new KernelArguments { ["base"] = 500.0, ["remise_pct"] = 15 }),
    ("850 TTC à 19,6 %", "tva_inverse", new KernelArguments { ["montant_ttc"] = 850.0, ["taux_tva"] = 0.196 }),
    ("Prix de 200 HT", "prix_ttc", new KernelArguments { ["base"] = 200.0 })
};
foreach (var (question, nom, arguments) in requetes)
    Console.WriteLine($"{question} -> {nom} -> {await kernel.InvokeAsync("mcp", nom, arguments, cancellationToken: ct)}");
});

Détail TVA pour 1000 HT -> calcule_tva -> {"content":[{"type":"text","text":"{\u0022ht\u0022:1000,\u0022tva\u0022:200,\u0022ttc\u0022:1200}"}],"structuredContent":{"ht":1000,"tva":200,"ttc":1200},"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


Prix de 500 HT, remise 15 % -> prix_ttc -> {"content":[{"type":"text","text":"{\u0022ht\u0022:425,\u0022ttc\u0022:510}"}],"structuredContent":{"ht":425,"ttc":510},"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


850 TTC à 19,6 % -> tva_inverse -> {"content":[{"type":"text","text":"{\u0022ttc\u0022:850,\u0022taux_tva\u0022:0.196,\u0022ht\u0022:710.7023}"}],"structuredContent":{"ttc":850,"taux_tva":0.196,"ht":710.7023},"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


Prix de 200 HT -> prix_ttc -> {"content":[{"type":"text","text":"{\u0022ht\u0022:200,\u0022ttc\u0022:240}"}],"structuredContent":{"ht":200,"ttc":240},"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}



warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.11.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Abstractions' matches identity 'Microsoft.Extensions.AI.Abstractions, Version=10.11.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' of 'Microsoft.Extensions.AI.Abstractions', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'Microsoft.Extensions.AI.Abstractions, Version=10.5.0.0, Culture=neutral, PublicKeyToken=31bf3856ad364e35' used by 'Microsoft.SemanticKernel.Ab

Les requêtes couvrent les trois outils de calcul ; les arguments sont connus à l’avance. Avec un véritable modèle de chat, SK pourrait lui déléguer la sélection, sous réserve de validation des arguments et des effets de bord.

### Exercice 3 — Analyser une chaîne

Appeler `analyse_chaine` via le client et lire les champs `nb_caracteres`, `nb_mots` et `premier_mot` de `structuredContent`. Comparez le schéma d’entrée découvert à celui du résultat.

In [11]:
string texteEx3 = "Le Model Context Protocol normalise les schemas d'outils.";
// TODO : appeler analyse_chaine avec la clé texte, puis lire son contenu JSON.
Console.WriteLine($"Exercice à compléter : analyser {texteEx3.Length} caractères");

Exercice à compléter : analyser 57 caractères


## 5. Exporter des fonctions SK vers MCP

Le serveur propose aussi un mode `--sk` : il crée un plugin SK, convertit ses fonctions par `AsAIFunctions(kernel)`, puis les enregistre via `McpServerTool.Create`. La fonction SK du carnet Python porte le nom historique `prix_ttc` mais calcule seulement le prix après remise, sans TVA. Nous conservons cette formule. Contrairement à une simple invocation directe SK, le test suivant traverse à nouveau le transport stdio.

In [12]:
await AvecClient(async (clientSk, ct) =>
{
    var exported = await clientSk.ListToolsAsync(cancellationToken: ct);
    Console.WriteLine($"Export SK : {exported.Count} outil(s) : {string.Join(", ", exported.Select(x => x.Name))}");
    Console.WriteLine("Schéma : " + exported[0].JsonSchema);
    var retour = await clientSk.CallToolAsync(exported[0].Name, new Dictionary<string, object> { ["base"] = 300.0, ["remise_pct"] = 20 }, cancellationToken: ct);
    var texte = JsonSerializer.SerializeToElement(retour).GetProperty("content")[0].GetProperty("text").GetString();
    if (double.Parse(texte, System.Globalization.CultureInfo.InvariantCulture) != 240.0) throw new Exception("Résultat export SK incorrect");
    Console.WriteLine("SK -> MCP -> client : " + JsonSerializer.Serialize(retour));
}, exporterSk: true);
Console.WriteLine("Session export SK fermée.");

Export SK : 1 outil(s) : prix_prix_ttc


Schéma : {"type":"object","condition":"Calcule le prix apr\u00E8s remise, sans TVA (nom historique).","properties":{"base":{"type":"number"},"remise_pct":{"description":"(default value: 0)","type":"integer"}},"required":["base"]}


SK -> MCP -> client : {"content":[{"type":"text","text":"240"}],"structuredContent":null,"isError":null,"_meta":{"io.modelcontextprotocol/serverInfo":{"name":"Sk08McpServer","version":"1.0.0.0"}},"resultType":"complete"}


Session export SK fermée.


Le serveur exporté et le premier serveur sont deux processus indépendants. Chaque session utilise une fermeture asynchrone en sortie de bloc, y compris après exception ; le transport dispose d’un délai de fermeture borné.

## À retenir

| Geste | Résultat |
|---|---|
| `CreateAsync` / `ListToolsAsync` | Initialiser et découvrir les contrats MCP |
| `CallToolAsync` | Exécuter à travers stdio et examiner `IsError` |
| `AsKernelFunction` | Consommer un outil MCP dans SK |
| `AsAIFunctions` / `McpServerTool.Create` | Exposer une fonction SK à un client MCP |

## Vérifiez votre compréhension

1. Pourquoi une erreur de validation se distingue-t-elle d’une panne de transport ?
2. Quel est le rôle du schéma d’entrée ?
3. Pourquoi le routage démontré ne prouve-t-il pas une décision de LLM ?

<details><summary>Éléments de réponse</summary>

1. La première est produite après réception d’un appel ; la seconde empêche l’échange.
2. Il décrit les noms, types et paramètres requis avant l’appel.
3. Le programme associe lui-même chaque demande à un outil et à ses arguments.
</details>

## Pour aller plus loin

Comparer le [notebook Python](08-SemanticKernel-MCP.ipynb), notamment FastMCP/Pydantic et `MCPStdioPlugin`, puis consulter la [spécification MCP](https://modelcontextprotocol.io/) et le [SDK C#](https://github.com/modelcontextprotocol/csharp-sdk).